# Modelling

First model to predict the number of rings of an abalone. The runs are tracked with MLflow (`mlflow ui` to compare them).

In [1]:
import mlflow
import numpy as np
import pandas as pd
from sklearn.compose import TransformedTargetRegressor
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import RepeatedKFold, cross_validate, train_test_split

In [2]:
df = pd.read_csv("../data/abalone.csv")
df = pd.get_dummies(df, columns=["Sex"], dtype=int)

X = df.drop(columns="Rings")
y = df["Rings"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [3]:
mlflow.set_experiment("abalone")

with mlflow.start_run(run_name="linear_regression"):
    model = LinearRegression()
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    rmse = mean_squared_error(y_test, y_pred) ** 0.5
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    mlflow.log_metrics({"rmse": rmse, "mae": mae, "r2": r2})
    mlflow.sklearn.log_model(model, name="model")

print(f"RMSE: {rmse:.2f}, MAE: {mae:.2f}, R2: {r2:.2f}")

2026/10/23 00:22:23 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


RMSE: 2.21, MAE: 1.59, R2: 0.55


In [4]:
params = {"n_estimators": 200, "max_depth": 10}

with mlflow.start_run(run_name="random_forest"):
    model = RandomForestRegressor(**params, random_state=42)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    rmse = mean_squared_error(y_test, y_pred) ** 0.5
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    mlflow.log_params(params)
    mlflow.log_metrics({"rmse": rmse, "mae": mae, "r2": r2})
    mlflow.sklearn.log_model(
        model, name="model", skops_trusted_types=["sklearn.tree._tree.Tree"]
    )

print(f"RMSE: {rmse:.2f}, MAE: {mae:.2f}, R2: {r2:.2f}")

2026/10/23 00:22:45 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


RMSE: 2.23, MAE: 1.56, R2: 0.54


Both models are close: RMSE around 2.2 rings and R2 around 0.55 on the test set. On this split the random forest is not better than the linear regression, but the test set is small (see the cross-validation below).

## Model selection with repeated cross-validation

The test set has only 836 rows, so the RMSE of a single split is noisy: the difference between the two models above is smaller than this noise. Here the models are compared with a 5-fold cross-validation repeated 5 times (same folds for every model).

I also try gradient boosting, which usually does a bit better than linear models on tabular data. On abalone, the TALENT benchmark ([Ye et al., arXiv:2407.00956](https://arxiv.org/abs/2407.00956), [results](https://github.com/LAMDA-Tabular/TALENT/blob/main/results/regression.md)) reports an RMSE of 2.24 for linear regression and 2.16 to 2.18 for tuned random forest and gradient boosting. The gradient boosting is trained on log(1 + rings), which gives less weight to the few very old abalones.

In [5]:
cv = RepeatedKFold(n_splits=5, n_repeats=5, random_state=42)
models = {
    "linear_regression": LinearRegression(),
    "random_forest": RandomForestRegressor(
        n_estimators=200, max_depth=10, random_state=42
    ),
    "gradient_boosting": TransformedTargetRegressor(
        regressor=HistGradientBoostingRegressor(
            learning_rate=0.05, max_iter=200, max_leaf_nodes=15, random_state=42
        ),
        func=np.log1p,
        inverse_func=np.expm1,
    ),
}

for name, model in models.items():
    scores = cross_validate(
        model,
        X,
        y,
        cv=cv,
        scoring=["neg_root_mean_squared_error", "neg_mean_absolute_error"],
    )
    rmse = -scores["test_neg_root_mean_squared_error"]
    mae = -scores["test_neg_mean_absolute_error"]
    with mlflow.start_run(run_name=f"cv_{name}"):
        mlflow.log_metrics(
            {"cv_rmse": rmse.mean(), "cv_rmse_std": rmse.std(), "cv_mae": mae.mean()}
        )
    print(f"{name}: RMSE {rmse.mean():.3f} (+- {rmse.std():.3f}), MAE {mae.mean():.3f}")

linear_regression: RMSE 2.211 (+- 0.120), MAE 1.584


random_forest: RMSE 2.162 (+- 0.104), MAE 1.521


gradient_boosting: RMSE 2.154 (+- 0.110), MAE 1.490


Gradient boosting is the best on both metrics: RMSE 2.15 and MAE 1.49, against 2.21 and 1.58 for the linear regression (-3% RMSE, -6% MAE), which is in line with the TALENT numbers. The random forest is almost as good, but its file is about 12 MB against 360 KB for the gradient boosting. So the training flow (`src/modelling/training.py`) now uses the gradient boosting.

Note: I tried a few hyperparameters (learning rate, number of iterations, number of leaves) on this same cross-validation, so the gain is maybe a bit optimistic. The +- is the standard deviation over the 25 folds.